[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/legalintermedia/Physics/blob/main/Simulaciones_Jupyter/Boutique_Fisica_Avanzada/Lattice_Campo_Gauge_U1.ipynb)



# Física No-Perturbativa: Lattice Gauge Theory U(1)
La **Cromodinámica Cuántica en el Retículo (Lattice QCD)** es la única forma matemática estricta conocida por la humanidad para resolver interacciones nucleares fuertes donde la constante de acoplamiento es tan grande que los diagramas de Feynman divergen al infinito.

Aquí implementamos una versión $U(1)$ en 2D (Electrodinámica Cuántica Compacta pura, sin fermiones) formulada por Kenneth Wilson.
Las variables dinámicas (el campo electromagnético cuántico) no viven en los nodos, sino en los **enlaces (links)** entre nodos, garantizando invarianza Gauge exacta. Calculamos la Acción de Wilson a través de las plaquetas (bucles elementales) y termalizamos el vacío con Monte Carlo.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Parámetros del Retículo (Lattice)
L = 16          # Tamaño de la red (L x L)
beta = 2.0      # Inverso de la constante de acoplamiento (beta ~ 1/e^2)
steps = 1000    # Pasos de Monte Carlo (Updates del campo Gauge)

# El campo Gauge U_mu(x) vive en los enlaces.
# links[x, y, 0] = enlace en dirección X partiendo del nodo (x,y)
# links[x, y, 1] = enlace en dirección Y partiendo del nodo (x,y)
# Los valores son ángulos (fases compactas) en [-pi, pi) representando exp(i*theta) en el grupo U(1).
links = np.random.uniform(-np.pi, np.pi, (L, L, 2))

def plaquette_action(x, y):
    """Calcula la suma de fases (flujo magnético) de la plaqueta 1x1 con esquina inferior izquierda en (x,y)"""
    U_bottom = links[x, y, 0]
    U_right  = links[(x+1)%L, y, 1]
    U_top    = links[x, (y+1)%L, 0]
    U_left   = links[x, y, 1]
    
    # El flujo en la plaqueta cerrada: Fondo + Derecha - Tope - Izquierda
    theta_p = U_bottom + U_right - U_top - U_left
    
    # La acción de Wilson (Parte Real de la traza: 1 - cos(theta))
    # Solo necesitamos la parte que cambia para el peso estadístico: -cos(theta)
    return -beta * np.cos(theta_p)

def local_action(x, y, mu, new_theta):
    """Calcula el cambio de acción si modificamos un único link"""
    # Un link en dirección mu (0 o 1) pertenece a dos plaquetas adyacentes.
    # Guardamos el link antiguo
    old_theta = links[x, y, mu]
    
    # Acción con el link antiguo
    S_old = plaquette_action(x, y) + (plaquette_action(x, (y-1)%L) if mu==0 else plaquette_action((x-1)%L, y))
    
    # Acción con el link nuevo
    links[x, y, mu] = new_theta
    S_new = plaquette_action(x, y) + (plaquette_action(x, (y-1)%L) if mu==0 else plaquette_action((x-1)%L, y))
    
    # Restaurar por si rechazamos
    links[x, y, mu] = old_theta
    
    return S_new - S_old

def metropolis_gauge_update():
    """Aplica algoritmo Metropolis a todo el retículo"""
    global links
    for x in range(L):
        for y in range(L):
            for mu in range(2):
                # Proponer un nuevo ángulo aleatorio U(1)
                theta_prop = links[x, y, mu] + np.random.uniform(-0.5, 0.5)
                theta_prop = (theta_prop + np.pi) % (2*np.pi) - np.pi
                
                # Diferencia de Acción de Wilson
                dS = local_action(x, y, mu, theta_prop)
                
                # Condición de aceptación
                if dS < 0 or np.random.rand() < np.exp(-dS):
                    links[x, y, mu] = theta_prop

# Termalizar el vacío cuántico
for _ in range(steps):
    metropolis_gauge_update()

# Calcular y visualizar la "Carga Magnética Topológica" o Fluctuaciones de Plaqueta
plaquettes = np.zeros((L, L))
for x in range(L):
    for y in range(L):
        theta_p = links[x,y,0] + links[(x+1)%L,y,1] - links[x,(y+1)%L,0] - links[x,y,1]
        plaquettes[x, y] = np.cos(theta_p)

plt.figure(figsize=(6, 6))
plt.imshow(plaquettes, cmap='magma', interpolation='nearest')
plt.title("Termalización del Vacío Cuántico (Lattice U(1) Gauge Field)")
plt.colorbar(label="Coseno del Flujo de Plaqueta $1 - \\frac{S_p}{\\beta}$")
plt.axis('off')
plt.show()
print("Esta imagen es el análogo matemático puro a cómo 'hierven' los gluones y quarks virtuales en el interior del protón.")